# ESG 텍스트마이닝 — 단계 I: 문서선별 및 전처리

분석계획서 v11 기준 | 오류점검 반영 버전

**실행 순서**: Cell 1 → 2 → 3 → 4 → 4b → 4c → 5 → 6 → 7 → 8 → 9 → 10 → 13 → 13-결과표 → 14 → 11 → 12

**사전 점검 결과 반영 사항**
- Cell 4b: 교수 JSONL이 이미 병합됨 → 건너뜀 처리
- Cell 4c: page_category NaN 290건(전부 ㄹ) → main으로 보완
- PROMO_PT = {ㄴ, ㅋ, ㅌ} (ㅁ은 전처리에서 main으로 확정)
- 절사 기준: 40,000자 (데이터 실제 최대치)
- 전화번호 패턴 47건 잔존 → TEXT_CLEAN_PATTERNS 재적용

## Cell 1. 라이브러리 임포트

In [31]:
import json
import re
import hashlib
import html
from pathlib import Path
from collections import Counter, defaultdict
from itertools import combinations

import pandas as pd
import openpyxl

try:
    from konlpy.tag import Okt
    OKT_AVAILABLE = True
    print("KoNLPy Okt 로드 완료")
except ImportError:
    OKT_AVAILABLE = False
    print("⚠ KoNLPy 없음 — pip install konlpy 후 재실행")

try:
    from tqdm.notebook import tqdm
    USE_TQDM = True
    print("tqdm 로드 완료")
except ImportError:
    def tqdm(x, **kwargs): return x
    USE_TQDM = False
    print("tqdm 없음 — 진행 표시 없이 실행")

print("\n라이브러리 임포트 완료")

KoNLPy Okt 로드 완료
tqdm 로드 완료

라이브러리 임포트 완료


## Cell 2. 경로 설정

In [32]:
# ── 입력 경로 ──────────────────────────────────────────────────────────────
DATA_PATH = Path(r"C:\Users\legen\Desktop\Lab Project\ESG\crawling\combined\ESG_crawled_final.jsonl")

# ── 출력 경로 ──────────────────────────────────────────────────────────────
OUTPUT_DIR = Path(r"C:\Users\legen\Desktop\Lab Project\ESG\v2\단계1_문서선별전처리")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"❌ 입력 파일 없음: {DATA_PATH}"
print(f"입력: {DATA_PATH}")
print(f"출력: {OUTPUT_DIR}")

입력: C:\Users\legen\Desktop\Lab Project\ESG\crawling\combined\ESG_crawled_final.jsonl
출력: C:\Users\legen\Desktop\Lab Project\ESG\v2\단계1_문서선별전처리


## Cell 3. 키워드 · stopword · 패턴 정의

In [33]:
# ══════════════════════════════════════════════════════════════════════════
# 3-1. ESG 키워드 사전  (법·물 FP 제외)
# ══════════════════════════════════════════════════════════════════════════
ESG_KEYWORDS = {
    'ESG_총괄': [
        'ESG 평가', 'ESG Rating', '지속가능경영', 'Sustainability Management',
        '지속가능성 보고서', 'Sustainability Report', '비재무정보', 'Non-financial Data',
        '통합보고서', 'Integrated Report', '임팩트 투자', 'Impact Investing',
        '책임투자', 'Responsible Investment', 'RI', '스튜어드십 코드', 'Stewardship Code',
        'TCFD', 'SASB', 'GRI기준', 'ISSB 기준', 'ESG KPI', 'ESG지표',
        '지속가능금융', 'Sustainable Finance', '녹색채권', 'Green Bond',
        '사회적 채권', 'Social Bond',
    ],
    'E': [
        '지속가능', 'sustainable', '지속가능성', 'sustainability',
        '그린', 'green', '기후', 'climate', '친환경', 'eco-friendly',
        '환경', 'environment', 'environmental', '탄소', 'carbon',
        '넷제로', 'net-zero', '온실가스', 'greenhouse gas', '가스', 'gas', 'GHG',
        'Scope 1', 'Scope 2', 'Scope 3', '에너지', 'energy',
        '태양광', '풍력', '대기전력', '자원', 'resource',
        '재활용', 'recycling', '업사이클링', 'upcycling', '분리배출',
        '재사용', 'reuse', '재판매', 'resale', '제품수선', 'repair',
        '오염', 'pollution', '미세먼지', '플라스틱',
        '순환경제', 'circular economy', '동물복지', 'animal welfare',
        # '물', 'water' → FP 제외 ('물'은 단독 제외, 'water'는 유지)
        'water', '폐기물', 'waste', '제로 웨이스트', 'zero waste',
        '생물다양성', 'biodiversity', '생태계', 'ecosystems',
        '윤리적', 'ethical', '화학물질', '유해물질',
        '절약', '감축', '관리', '대체', '행동', '복지', '실천', '보호', '보전', '감시',
    ],
    'S': [
        '사회적 책임', 'social responsibility', '책임', 'responsibility',
        '책임있는', 'responsible', '노동', 'labor', '직원', 'employee',
        '권리', 'rights', '복지', 'welfare', '참여', 'engagement',
        '소비자', 'consumer', '소비', 'consumption', '행동', 'behavior',
        '보호', 'protection', '신뢰', 'trust',
        '인권', 'human rights', '인적 자본', 'human capital',
        '다양성', 'diversity', '평등', '형평성', 'equity',
        '포용성', 'inclusion', '포용적', 'inclusive',
        '지역사회', 'community', '개발', '활동', '의식', '공헌',
        '건강', 'health', '안전', 'safety', '위생', 'sanitation',
        '고객', 'customer', '빈곤', 'poverty', '기아', 'hunger',
        '식량', 'food', '안보', 'security',
        '교육', 'education', '역량', 'upskilling', '훈련', 'training',
        '성평등', 'gender equity', '여성', 'women',
        '성 격차', 'gender disparities', '불평등', 'inequity',
        '공정', 'fair', '대우', 'treatment', '임금', 'wages',
        '기회', 'opportunity', '일자리', 'work', '경제 성장', 'economic growth',
        '비재무', 'non-financial', '제품 품질', 'product quality',
        '제품수명주기', 'product life cycle',
        '공급망 관리', 'Supply Chain Management',
        '협력사 관리', 'Vendor Management',
        '이해관계자', 'Stakeholders', '사회적 가치', 'Social Value',
        '관계', '돌봄', '웰빙', 'well-being', '삶의 질',
        '윤리', 'ethic', '개인정보 보호', 'data privacy',
        '디지털 시민성', 'digital citizenship',
    ],
    'G': [
        'ESG 환경경영', 'ESG 위원회', '기업 가치', 'firm value',
        '주주 가치', 'shareholder value',
        '투명성', 'transparency', '공시', 'disclosure', '표시제도',
        '윤리', 'ethic', 'ethical', '반부패', 'anti-corruption',
        '뇌물방지', 'Anti-corruption',
        # '법규 준수', 'legal compliance' → '법' FP 제외, legal compliance는 영문이므로 유지
        'legal compliance', '이사회', 'Board of Directors',
        '독립성', 'Board Independence', '감사위원회', 'Audit Committee',
        '경영진 보상', 'Executive Compensation',
        '주주권', 'Shareholder Rights',
        '파트너쉽', 'partnership', '협력', 'collaboration',
        '기업지배구조', 'Corporate Governance',
        '가계관리', '가계재무', '소비계획', '의사결정 기준',
        '규칙', '정보 활용', '정보 판단',
    ],
}

# FP 제외 키워드 (str.contains 선별에서 제외)
FP_EXCLUDE_KW = {'법', '물'}

# 선별에 사용할 플랫 키워드 리스트 (FP 제외 후)
ALL_KW_FLAT = []
for cat, kws in ESG_KEYWORDS.items():
    for kw in kws:
        if kw not in FP_EXCLUDE_KW:
            ALL_KW_FLAT.append((cat, kw))
print(f"ESG 키워드 총 {len(ALL_KW_FLAT)}개 (법·물 제외)")

# ══════════════════════════════════════════════════════════════════════════
# 3-2. 행정 stopword 146개
# ══════════════════════════════════════════════════════════════════════════
ADMIN_STOPWORDS_RAW = [
    # 1글자 UI 잔재
    '홈', '동', '관', '상', '순', '외', '전', '중', '후', '형', '당',
    # 행정·공지 일반
    '공지사항', '공지', '안내', '알림', '게시판', '첨부', '첨부파일',
    '다운로드', '업로드', '제출', '제출기한', '제출방법', '신청',
    '신청기간', '신청방법', '신청서', '서류', '양식', '확인', '문의',
    '접수', '마감', '기간', '기한', '일정', '일자', '날짜', '시간',
    '일시', '장소', '방법', '절차', '기준', '대상', '해당', '내용',
    '사항', '경우', '관련', '수행', '운영', '진행', '실시', '추진',
    '개최', '시행', '적용', '해당자', '이상', '이하', '초과', '미만',
    '참고', '참조', '참조바람', '바람', '바랍니다', '합니다', '합니다.',
    '드립니다', '드립니다.',
    # 행정 단위
    '학과', '학부', '대학원', '학과장', '담당', '담당자', '교수님',
    '조교', '사무실', '사무처', '행정실', '부서', '팀', '실',
    # 게시판 UI
    '번호', '제목', '작성자', '작성일', '조회수', '조회', '댓글',
    '이전글', '다음글', '목록', '처음', '끝', '페이지', '검색',
    '등록', '수정', '삭제', '저장', '취소', '완료', '닫기',
    # 학사 행정
    '학점', '이수', '수강', '수업', '강의', '강의계획', '계획',
    '성적', '출석', '시험', '과제', '레포트', '발표', '평가',
    '학번', '이름', '연락처', '이메일', '전화', '팩스',
    # 일반 고빈도 형식어
    '및', '또는', '혹은', '등', '각', '본', '해당', '위한', '위해',
    '통해', '통한', '위하여', '으로', '에서', '에게', '에게서',
    '대하여', '대한', '관하여', '관한',
    # 숫자·기호 잔재
    '년도', '년', '월', '일', '분기', '기간', '학기', '회', '개',
    '명', '건', '원', '만원', '억원',
    # 기타 UI 잔재
    '바로가기', '상세보기', '더보기', '전체보기', '링크', '클릭',
    '닫기', '열기', '펼치기', '접기',
    
    # 학술·논문 행정어 (논문 목록 텍스트 잔재)
    '논문', '학술', '학술지', '학위', '일반', '학회', '춘계', '추계',
    '대회', '국제', '대한민국', '우수', '저서', '역서', '편집', '위원',
    '회지', '포스터', '주년', '기념',

    # 교과과정 행정어 (교육과정표 잔재)
    '전공', '공선택', '교과목', '학년', '필수', '학점', '과목',
    '분반', '교양', '이수구분', '학사', '석사', '박사', '통합과정',

    # 교수 프로필 잔재
    '교수', '소개', '명예', '취임', '퇴임', '관심', '분야', '박사과정',
    '석사졸업', '박사졸업', '졸업논문',

    # 기관명 파편 (형태소 오류)
    '그램',     # 프로그램 → '프로', '그램' 으로 분리됨
    '회수',     # '작성_회수' 등 교과목 수 관련 잔재
    '스템',     # 시스템 → '시', '스템' 으로 분리됨
    '자학',     # 소비자학 → '소비', '자학' 으로 분리됨
]
ADMIN_STOPWORDS = set(ADMIN_STOPWORDS_RAW)
print(f"행정 stopword: {len(ADMIN_STOPWORDS)}개")

# ══════════════════════════════════════════════════════════════════════════
# 3-3. 텍스트 정제 패턴 (전처리 후에도 잔존하는 패턴 재처리)
# ══════════════════════════════════════════════════════════════════════════
TEXT_CLEAN_PATTERNS = [
    (re.compile(r'\d{2,4}[-. ]\d{3,4}[-. ]\d{4}'), ' '),  # 전화번호
    (re.compile(r'[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}'), ' '),  # 이메일
    (re.compile(r'https?://\S+'), ' '),  # URL
    (re.compile(r'&[a-zA-Z#0-9]+;'), ' '),  # HTML 엔티티
    (re.compile(r'[-=_]{4,}'), ' '),  # 연속 특수기호
    (re.compile(r'[ \t]+'), ' '),  # 연속 공백
    (re.compile(r'\n{3,}'), '\n\n'),  # 연속 개행
]

def apply_text_clean(text: str) -> str:
    if not isinstance(text, str):
        text = str(text)
    text = html.unescape(text)
    for pat, rep in TEXT_CLEAN_PATTERNS:
        text = pat.sub(rep, text)
    return text.strip()

# ══════════════════════════════════════════════════════════════════════════
# 3-4. n-gram 후처리 상수
# ══════════════════════════════════════════════════════════════════════════
# 형태소 분석 오류 치환 (결과 bigram에 적용)
NGRAM_REPLACE = {
    '소비_자학': '소비자학',
    '소비자_학과': '소비자학과',
    '지속_가능': '지속가능',
    '환경_오': '환경오',
    '양성평등': '성평등'
}

# UI 잔재 bigram exact 제거
NGRAM_STOP_EXACT = {
    '상세_보기', '강의_계획', '계획_상세', '다음_글', '이전_글',
    '목록_보기', '바로_가기', '전체_보기', '첨부_파일',

}

# 문장 분리 정규식
SENT_SPLIT_RE = re.compile(r'(?<=[.!?。])[\s\n]+')


# ══════════════════════════════════════════════════════════════════════════
# 3-5. Track B 복원을 위해 보존할 1글자 형태소
# ══════════════════════════════════════════════════════════════════════════
# 예: 사회_적_책임, 성_격차, 비_재무_정보, 주주_권
NGRAM_KEEP_1CHAR = {
    '적', '성', '비', '권'
}


# ══════════════════════════════════════════════════════════════════════════
# 3-6. 영어/논문 메타 n-gram 잡음 제거 규칙
# ══════════════════════════════════════════════════════════════════════════
# 목적:
# 1) Track B exact 후보를 점검할 때 Journal/SCIE/Scopus/KCI류 메타 잡음을 줄임
# 2) 최종 ESG 카운트 왜곡 방지라기보다 QA/디버깅 가독성 개선용

NGRAM_META_BLOCK_TOKENS = {
    'JOURNAL', 'SCIE', 'SSCI', 'AHCI', 'A&HCI', 'SCOPUS', 'KCI', 'KCI등재',
    'DCOLL', 'VOL', 'NO', 'ISSUE', 'PP', 'DOI', 'ORCID', 'ISSN', 'ISBN',
    'AUTHOR', 'KEYWORD', 'KEYWORDS',

    'JSP', 'LAYOUT', 'UNKNOWN', 'WEB', 'COM', 'COP', 'SITE', 'INF',
    'ARTCLVIEW', 'IN', 'THE', 'of', 'the', 'Korean', 'and',

    'OF', 'THE', 'AND', 'IN', 'ON', 'FOR', 'TO', 'WITH',

    'CONFERENCE', 'SYMPOSIUM', 'MEETING', 'ANNUAL', 'SOCIETY',
    'RESEARCH', 'REVIEW', 'DEPARTMENT', 'COLLEGE', 'UNIVERSITY',
    'INTERNATIONAL', 'NATIONAL', 'KOREAN', 'KOREA', 'SOUTH',

    'TOP', 'MENU', 'QUICK', 'DOWNLOAD', 'NOTICE', 'DETAIL', 'VIEW',
    'COPYRIGHT', 'RIGHT', 'RIGHTS', 'RESERVED', 'ENGLISH',

    'FNCTID', 'FNCTNO', 'BBS', 'JW', 'MS', 'WT'
}

NGRAM_META_BLOCK_EXACT = {
    'JOURNAL_OF', 'OF_THE', 'SCIE_SCOPUS', 'SCOPUS_DCOLL',
    'NO_SCIE', 'NO_KCI', 'KCI_DCOLL', 'AUTHOR_KEYWORD',
    'SCIE_SCOPUS_DCOLL',
    'NO_SCIE_SCOPUS',
    'NO_KCI_DCOLL',
    'SCOPUS_KCI_DCOLL',
    'SCIE_SCOPUS_KCI',

    'WEB_INF_JSP',
    'INF_JSP_WEB',
    'JSP_WEB_COM',
    'WEB_COM_COP',
    'COM_COP_SITE',
    'COP_SITE_LAYOUT',
    'SITE_LAYOUT_JSP',

    'ARTCLVIEW_DO_LAYOUT',
    'DO_LAYOUT_UNKNOWN',

    'INTERNATIONAL_JOURNAL_OF',
    'JOURNAL_OF_THE',
    'KOREAN_SOCIETY_OF',
    'UNIVERSITY_OF',
    'EFFECTS_OF',
}

def normalize_meta_token(tok: str) -> str:
    tok = str(tok).strip()
    tok = tok.replace('&', '')
    tok = re.sub(r'[^0-9A-Za-z가-힣]+', '', tok)
    return tok.upper()

def is_meta_noise_ngram(ngram: str) -> bool:
    """
    영어/논문 실적 메타 n-gram 여부 판정
    - 예: Journal_of, SCIE_Scopus, no_KCI, pp_123 등
    - ESG/GRI/TCFD 같은 Track C 약어 보완 대상은 여기서 일괄 제거하지 않음
    """
    if not isinstance(ngram, str) or not ngram.strip():
        return True

    if ngram.upper() in NGRAM_META_BLOCK_EXACT:
        return True

    parts = [p for p in str(ngram).split('_') if p]
    if not parts:
        return True

    norm_parts = [normalize_meta_token(p) for p in parts]

    # 1) 토큰 단위 exact 차단
    if any(p in NGRAM_META_BLOCK_TOKENS for p in norm_parts):
        return True

    # 2) 대표적인 영문 메타 패턴 차단
    joined = '_'.join(norm_parts)
    meta_patterns = [
        r'^(JOURNAL|SCIE|SSCI|AHCI|SCOPUS|KCI|DCOLL|VOL|NO|ISSUE|PP|DOI|ORCID|ISSN|ISBN)(_|$)',
        r'(^|_)(JOURNAL|SCIE|SSCI|AHCI|SCOPUS|KCI|DCOLL|VOL|NO|ISSUE|PP|DOI|ORCID|ISSN|ISBN)$',
    ]
    if any(re.search(pat, joined) for pat in meta_patterns):
        return True

    # 3) 전부 영문/숫자 토큰이고 메타 토큰이 섞여 있으면 차단
    ascii_like = [bool(re.fullmatch(r'[A-Z0-9]+', p)) for p in norm_parts]
    meta_like = [p in NGRAM_META_BLOCK_TOKENS for p in norm_parts]
    if all(ascii_like) and sum(meta_like) >= 1:
        return True
    
    # 4) 전부 영어인 n-gram은 기본 제거
    #    단, ESG 키워드 파일에 있는 영어 canonical 표현은 예외 허용
    #    개선되지 않을 시 주석 처리
    if is_all_english_ngram(ngram):
        if compact_alpha_ngram(ngram) not in ALLOWED_ENGLISH_ESG_COMPACT:
            return True

    return False


# ── Track B에서 허용할 영어 ESG canonical (공백/underscore 제거형) ─────────
ALLOWED_ENGLISH_ESG_COMPACT = {
    'socialresponsibility',
    'humancapital',
    'economicgrowth',
    'animalwelfare',
    'circulareconomy',
    'socialvalue',
    'humanrights',
    'productquality',
    'sustainablefinance',
    'sustainabilitymanagement',
    'sustainabilityreport',
    'greenbond',
    'socialbond',
    'impactinvesting',
    'responsibleinvestment',
    'stewardshipcode',
    'integratedreport',
    'nonfinancialdata',
    'supplychainmanagement',
    'vendormanagement',
    'dataprivacy',
    'digitalcitizenship',
    'boardofdirectors',
    'boardindependence',
    'auditcommittee',
    'executivecompensation',
    'corporategovernance',
    'shareholderrights'
}

def compact_alpha_ngram(ngram: str) -> str:
    return re.sub(r'[_\s\-]+', '', str(ngram)).lower()

def is_all_english_ngram(ngram: str) -> bool:
    parts = [p for p in str(ngram).split('_') if p]
    if not parts:
        return False
    return all(bool(re.fullmatch(r'[A-Za-z]+', p)) for p in parts)


print("\nCell 3 완료 — 모든 상수 정의")

# ══════════════════════════════════════════════════════════════════════════
# 3-8. 한글-영문 ESG 개념 매핑 자산 (단계 II concept 병합용)
# ══════════════════════════════════════════════════════════════════════════
# 원칙:
# 1) raw 집계는 그대로 유지한다.
# 2) concept 집계는 한글-영문 대응어를 사후 병합하기 위한 자산으로만 사용한다.
# 3) 상위/하위 개념은 자동 병합하지 않고, 키워드 문서와 현재 Cell 3 키워드에 있는
#    명시적 한글-영문 대응어만 매핑한다.

def normalize_alias_key(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r'[_\s\-]+', '', text)
    return text.lower()

ESG_CONCEPT_SPECS = [
    # ESG_총괄
    {'category': 'ESG_총괄', 'canonical_ko': 'ESG 평가', 'aliases': ['ESG 평가', 'ESG Rating']},
    {'category': 'ESG_총괄', 'canonical_ko': '지속가능경영', 'aliases': ['지속가능경영', 'Sustainability Management']},
    {'category': 'ESG_총괄', 'canonical_ko': '지속가능성 보고서', 'aliases': ['지속가능성 보고서', 'Sustainability Report']},
    {'category': 'ESG_총괄', 'canonical_ko': '비재무정보', 'aliases': ['비재무정보', 'Non-financial Data']},
    {'category': 'ESG_총괄', 'canonical_ko': '통합보고서', 'aliases': ['통합보고서', 'Integrated Report']},
    {'category': 'ESG_총괄', 'canonical_ko': '임팩트 투자', 'aliases': ['임팩트 투자', 'Impact Investing']},
    {'category': 'ESG_총괄', 'canonical_ko': '책임투자', 'aliases': ['책임투자', 'Responsible Investment', 'RI']},
    {'category': 'ESG_총괄', 'canonical_ko': '스튜어드십 코드', 'aliases': ['스튜어드십 코드', 'Stewardship Code']},
    {'category': 'ESG_총괄', 'canonical_ko': '지속가능금융', 'aliases': ['지속가능금융', 'Sustainable Finance']},
    {'category': 'ESG_총괄', 'canonical_ko': '녹색채권', 'aliases': ['녹색채권', 'Green Bond']},
    {'category': 'ESG_총괄', 'canonical_ko': '사회적 채권', 'aliases': ['사회적 채권', 'Social Bond']},

    # Environmental
    {'category': 'E', 'canonical_ko': '지속가능', 'aliases': ['지속가능', 'sustainable']},
    {'category': 'E', 'canonical_ko': '지속가능성', 'aliases': ['지속가능성', 'sustainability']},
    {'category': 'E', 'canonical_ko': '그린', 'aliases': ['그린', 'green']},
    {'category': 'E', 'canonical_ko': '기후', 'aliases': ['기후', 'climate']},
    {'category': 'E', 'canonical_ko': '친환경', 'aliases': ['친환경', 'eco-friendly']},
    {'category': 'E', 'canonical_ko': '환경', 'aliases': ['환경', 'environment', 'environmental']},
    {'category': 'E', 'canonical_ko': '탄소', 'aliases': ['탄소', 'carbon']},
    {'category': 'E', 'canonical_ko': '넷제로', 'aliases': ['넷제로', 'net-zero']},
    {'category': 'E', 'canonical_ko': '온실가스', 'aliases': ['온실가스', 'greenhouse gas']},
    {'category': 'E', 'canonical_ko': '가스', 'aliases': ['가스', 'gas']},
    {'category': 'E', 'canonical_ko': '에너지', 'aliases': ['에너지', 'energy']},
    {'category': 'E', 'canonical_ko': '자원', 'aliases': ['자원', 'resource']},
    {'category': 'E', 'canonical_ko': '재활용', 'aliases': ['재활용', 'recycling']},
    {'category': 'E', 'canonical_ko': '업사이클링', 'aliases': ['업사이클링', 'upcycling']},
    {'category': 'E', 'canonical_ko': '재사용', 'aliases': ['재사용', 'reuse']},
    {'category': 'E', 'canonical_ko': '재판매', 'aliases': ['재판매', 'resale']},
    {'category': 'E', 'canonical_ko': '제품수선', 'aliases': ['제품수선', 'repair']},
    {'category': 'E', 'canonical_ko': '오염', 'aliases': ['오염', 'pollution']},
    {'category': 'E', 'canonical_ko': '순환경제', 'aliases': ['순환경제', 'circular economy']},
    {'category': 'E', 'canonical_ko': '동물복지', 'aliases': ['동물복지', 'animal welfare']},
    {'category': 'E', 'canonical_ko': '폐기물', 'aliases': ['폐기물', 'waste']},
    {'category': 'E', 'canonical_ko': '제로 웨이스트', 'aliases': ['제로 웨이스트', 'zero waste']},
    {'category': 'E', 'canonical_ko': '생물다양성', 'aliases': ['생물다양성', 'biodiversity']},
    {'category': 'E', 'canonical_ko': '생태계', 'aliases': ['생태계', 'ecosystems']},
    {'category': 'E', 'canonical_ko': '윤리적', 'aliases': ['윤리적', 'ethical']},

    # Social
    {'category': 'S', 'canonical_ko': '사회적 책임', 'aliases': ['사회적 책임', 'social responsibility']},
    {'category': 'S', 'canonical_ko': '책임', 'aliases': ['책임', 'responsibility']},
    {'category': 'S', 'canonical_ko': '책임있는', 'aliases': ['책임있는', 'responsible']},
    {'category': 'S', 'canonical_ko': '노동', 'aliases': ['노동', 'labor']},
    {'category': 'S', 'canonical_ko': '직원', 'aliases': ['직원', 'employee']},
    {'category': 'S', 'canonical_ko': '권리', 'aliases': ['권리', 'rights']},
    {'category': 'S', 'canonical_ko': '복지', 'aliases': ['복지', 'welfare']},
    {'category': 'S', 'canonical_ko': '참여', 'aliases': ['참여', 'engagement']},
    {'category': 'S', 'canonical_ko': '소비자', 'aliases': ['소비자', 'consumer']},
    {'category': 'S', 'canonical_ko': '소비', 'aliases': ['소비', 'consumption']},
    {'category': 'S', 'canonical_ko': '행동', 'aliases': ['행동', 'behavior']},
    {'category': 'S', 'canonical_ko': '보호', 'aliases': ['보호', 'protection']},
    {'category': 'S', 'canonical_ko': '신뢰', 'aliases': ['신뢰', 'trust']},
    {'category': 'S', 'canonical_ko': '인권', 'aliases': ['인권', 'human rights']},
    {'category': 'S', 'canonical_ko': '인적 자본', 'aliases': ['인적 자본', 'human capital']},
    {'category': 'S', 'canonical_ko': '다양성', 'aliases': ['다양성', 'diversity']},
    {'category': 'S', 'canonical_ko': '형평성', 'aliases': ['형평성', 'equity']},
    {'category': 'S', 'canonical_ko': '포용성', 'aliases': ['포용성', 'inclusion']},
    {'category': 'S', 'canonical_ko': '포용적', 'aliases': ['포용적', 'inclusive']},
    {'category': 'S', 'canonical_ko': '지역사회', 'aliases': ['지역사회', 'community']},
    {'category': 'S', 'canonical_ko': '건강', 'aliases': ['건강', 'health']},
    {'category': 'S', 'canonical_ko': '안전', 'aliases': ['안전', 'safety']},
    {'category': 'S', 'canonical_ko': '위생', 'aliases': ['위생', 'sanitation']},
    {'category': 'S', 'canonical_ko': '고객', 'aliases': ['고객', 'customer']},
    {'category': 'S', 'canonical_ko': '빈곤', 'aliases': ['빈곤', 'poverty']},
    {'category': 'S', 'canonical_ko': '기아', 'aliases': ['기아', 'hunger']},
    {'category': 'S', 'canonical_ko': '식량', 'aliases': ['식량', 'food']},
    {'category': 'S', 'canonical_ko': '안보', 'aliases': ['안보', 'security']},
    {'category': 'S', 'canonical_ko': '교육', 'aliases': ['교육', 'education']},
    {'category': 'S', 'canonical_ko': '역량', 'aliases': ['역량', 'upskilling']},
    {'category': 'S', 'canonical_ko': '훈련', 'aliases': ['훈련', 'training']},
    {'category': 'S', 'canonical_ko': '성평등', 'aliases': ['성평등', 'gender equity']},
    {'category': 'S', 'canonical_ko': '여성', 'aliases': ['여성', 'women']},
    {'category': 'S', 'canonical_ko': '성 격차', 'aliases': ['성 격차', 'gender disparities']},
    {'category': 'S', 'canonical_ko': '불평등', 'aliases': ['불평등', 'inequity']},
    {'category': 'S', 'canonical_ko': '공정', 'aliases': ['공정', 'fair']},
    {'category': 'S', 'canonical_ko': '대우', 'aliases': ['대우', 'treatment']},
    {'category': 'S', 'canonical_ko': '임금', 'aliases': ['임금', 'wages']},
    {'category': 'S', 'canonical_ko': '기회', 'aliases': ['기회', 'opportunity']},
    {'category': 'S', 'canonical_ko': '일자리', 'aliases': ['일자리', 'work']},
    {'category': 'S', 'canonical_ko': '경제 성장', 'aliases': ['경제 성장', 'economic growth']},
    {'category': 'S', 'canonical_ko': '비재무', 'aliases': ['비재무', 'non-financial']},
    {'category': 'S', 'canonical_ko': '제품 품질', 'aliases': ['제품 품질', 'product quality']},
    {'category': 'S', 'canonical_ko': '제품수명주기', 'aliases': ['제품수명주기', 'product life cycle']},
    {'category': 'S', 'canonical_ko': '공급망 관리', 'aliases': ['공급망 관리', 'Supply Chain Management']},
    {'category': 'S', 'canonical_ko': '협력사 관리', 'aliases': ['협력사 관리', 'Vendor Management']},
    {'category': 'S', 'canonical_ko': '이해관계자', 'aliases': ['이해관계자', 'Stakeholders']},
    {'category': 'S', 'canonical_ko': '사회적 가치', 'aliases': ['사회적 가치', 'Social Value']},
    {'category': 'S', 'canonical_ko': '웰빙', 'aliases': ['웰빙', 'well-being']},
    {'category': 'S', 'canonical_ko': '윤리', 'aliases': ['윤리', 'ethic']},
    {'category': 'S', 'canonical_ko': '개인정보 보호', 'aliases': ['개인정보 보호', 'data privacy']},
    {'category': 'S', 'canonical_ko': '디지털 시민성', 'aliases': ['디지털 시민성', 'digital citizenship']},

    # Governance
    {'category': 'G', 'canonical_ko': '기업 가치', 'aliases': ['기업 가치', 'firm value']},
    {'category': 'G', 'canonical_ko': '주주 가치', 'aliases': ['주주 가치', 'shareholder value']},
    {'category': 'G', 'canonical_ko': '투명성', 'aliases': ['투명성', 'transparency']},
    {'category': 'G', 'canonical_ko': '공시', 'aliases': ['공시', 'disclosure']},
    {'category': 'G', 'canonical_ko': '반부패', 'aliases': ['반부패', 'anti-corruption']},
    {'category': 'G', 'canonical_ko': '뇌물방지', 'aliases': ['뇌물방지', 'Anti-corruption']},
    {'category': 'G', 'canonical_ko': '이사회', 'aliases': ['이사회', 'Board of Directors']},
    {'category': 'G', 'canonical_ko': '독립성', 'aliases': ['독립성', 'Board Independence']},
    {'category': 'G', 'canonical_ko': '감사위원회', 'aliases': ['감사위원회', 'Audit Committee']},
    {'category': 'G', 'canonical_ko': '경영진 보상', 'aliases': ['경영진 보상', 'Executive Compensation']},
    {'category': 'G', 'canonical_ko': '주주권', 'aliases': ['주주권', 'Shareholder Rights']},
    {'category': 'G', 'canonical_ko': '파트너쉽', 'aliases': ['파트너쉽', 'partnership']},
    {'category': 'G', 'canonical_ko': '협력', 'aliases': ['협력', 'collaboration']},
    {'category': 'G', 'canonical_ko': '기업지배구조', 'aliases': ['기업지배구조', 'Corporate Governance']},
]

ESG_CONCEPT_MAP = {
    spec['canonical_ko']: list(spec['aliases'])
    for spec in ESG_CONCEPT_SPECS
}

CONCEPT_TO_CATEGORY = {
    spec['canonical_ko']: spec['category']
    for spec in ESG_CONCEPT_SPECS
}

ALIAS_TO_CONCEPT = {}
for spec in ESG_CONCEPT_SPECS:
    canonical_ko = spec['canonical_ko']
    for alias in set(spec['aliases']) | {canonical_ko}:
        key = normalize_alias_key(alias)
        if key:
            ALIAS_TO_CONCEPT[key] = canonical_ko

concept_rows = []
for spec in ESG_CONCEPT_SPECS:
    canonical_ko = spec['canonical_ko']
    for alias in sorted(set(spec['aliases']) | {canonical_ko}):
        concept_rows.append({
            'canonical_ko': canonical_ko,
            'category': spec['category'],
            'alias': alias,
            'alias_key': normalize_alias_key(alias),
            'is_english_alias': bool(re.search(r'[A-Za-z]', alias)),
        })

concept_map_df = pd.DataFrame(concept_rows).sort_values(
    ['category', 'canonical_ko', 'is_english_alias', 'alias']
).reset_index(drop=True)

print(f"개념 매핑 수: {len(ESG_CONCEPT_MAP)}개")
print(f"alias key 수: {len(ALIAS_TO_CONCEPT)}개")


# ══════════════════════════════════════════════════════════════════════════
# 3-9. Track A 영어 unigram raw 자산 (일관성 보완)
# ══════════════════════════════════════════════════════════════════════════
# 원칙:
# 1) Track A raw_ko는 기존 Okt nouns_filtered 구조를 유지한다.
# 2) Track A raw_en은 영어 ESG unigram을 regex 기반으로 추출한다.
# 3) Track A concept는 raw_ko + raw_en을 대응어 기준으로 병합하는 후처리 레이어를 전제로 한다.
# 4) Track A/B/C concept는 트랙 내부에서만 병합하며, 트랙 간 재병합은 하지 않는다.

ENGLISH_UNIGRAM_RE = re.compile(r"[A-Za-z]+(?:-[A-Za-z]+)*")

def is_single_english_unigram(alias: str) -> bool:
    alias = str(alias).strip()
    return bool(re.fullmatch(r"[A-Za-z]+(?:-[A-Za-z]+)*", alias))

TRACKA_ENGLISH_UNIGRAM_ALIASES = sorted({
    str(alias).lower()
    for spec in ESG_CONCEPT_SPECS
    if (' ' not in spec['canonical_ko'] and '_' not in spec['canonical_ko'])
    for alias in (set(spec['aliases']) | {spec['canonical_ko']})
    if is_single_english_unigram(alias)
})

TRACKA_ENGLISH_UNIGRAM_ALIAS_KEYS = {
    normalize_alias_key(alias) for alias in TRACKA_ENGLISH_UNIGRAM_ALIASES
}

print(f"Track A 영어 unigram alias 수: {len(TRACKA_ENGLISH_UNIGRAM_ALIASES)}개")


ESG 키워드 총 257개 (법·물 제외)
행정 stopword: 216개

Cell 3 완료 — 모든 상수 정의
개념 매핑 수: 102개
alias key 수: 205개
Track A 영어 unigram alias 수: 67개


## Cell 4. 기존 JSONL 로드

In [34]:
records = []
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"로드 완료: {len(df)}건")
print(f"컬럼: {list(df.columns)}")
print(f"\npage_type 분포:")
print(df['page_type'].value_counts().sort_index())
print(f"\npage_category 분포 (NaN 포함):")
print(df['page_category'].value_counts(dropna=False))

로드 완료: 1158건
컬럼: ['대학명', '단과대학', '계열', '메뉴명', 'page_type', 'page_category', '비고', 'source_url', 'collected_url', 'text', 'text_length', 'domain', '설립구분', '성별구분', 'page_type_label', 'text_len_bin']

page_type 분포:
page_type
ㄱ    193
ㄴ      5
ㄷ    143
ㄹ    391
ㅁ     51
ㅂ     49
ㅅ     15
ㅇ      7
ㅈ     75
ㅊ     13
ㅋ    195
ㅌ     21
Name: count, dtype: int64

page_category 분포 (NaN 포함):
page_category
main     634
None     290
promo    234
Name: count, dtype: int64


## Cell 4b. 교수 데이터 통합

**⚠ 사전 점검 결과: 교수 JSONL이 이미 ESG_crawled_final.jsonl에 병합됨**  
page_category=NaN인 290건이 전부 교수페이지(page_type=ㄹ) 데이터.  
→ 이 셀은 건너뜀. Cell 4c에서 NaN 보완만 수행.

In [35]:
# 교수 JSONL 이미 병합 여부 확인
pc_nan = df[df['page_category'].isna()]
print(f"page_category NaN: {len(pc_nan)}건")
if len(pc_nan) > 0:
    pt_dist = pc_nan['page_type'].value_counts().to_dict()
    print(f"  page_type 분포: {pt_dist}")
    if set(pt_dist.keys()) == {'ㄹ'}:
        print("  ✓ 전부 ㄹ — 교수페이지 JSONL이 이미 병합된 행")
        print("  → Cell 4b 건너뜀. Cell 4c에서 page_category 보완 예정")
    else:
        print("  ⚠ ㄹ 외 다른 page_type에도 NaN 존재 — 확인 필요")
else:
    print("  ✓ page_category NaN 없음 — Cell 4b 불필요")

page_category NaN: 290건
  page_type 분포: {'ㄹ': 290}
  ✓ 전부 ㄹ — 교수페이지 JSONL이 이미 병합된 행
  → Cell 4b 건너뜀. Cell 4c에서 page_category 보완 예정


## Cell 4c. NaN 보완 + TEXT_CLEAN 재적용 + 40,000자 절사

In [36]:
CHAR_LIMIT = 40000  # 사용자 설정값 (계획서 표기 30000이나 실제 데이터 기준)
PROMO_PT = {'ㄴ', 'ㅋ', 'ㅌ'}  # 전처리 결과와 일치 (ㅁ=main 확정)

# ── 1. page_category NaN 보완 ──────────────────────────────────────────
before_nan = df['page_category'].isna().sum()
df['page_category'] = df.apply(
    lambda r: ('promo' if r['page_type'] in PROMO_PT else 'main')
    if pd.isna(r['page_category'])
    else r['page_category'],
    axis=1
)
after_nan = df['page_category'].isna().sum()
print(f"page_category NaN 보완: {before_nan}건 → {after_nan}건")

# ── 2. crawl_dual 컬럼 생성 ────────────────────────────────────────────
df['crawl_dual'] = df['page_category']  # main / promo
print(f"crawl_dual 분포:\n{df['crawl_dual'].value_counts()}")

# ── 3. TEXT_CLEAN 재적용 (전화번호 등 잔존 패턴 처리) ─────────────────
before_clean = df['text'].apply(lambda x: bool(re.search(r'\d{2,4}[-. ]\d{3,4}[-. ]\d{4}', str(x)))).sum()
df['text'] = df['text'].apply(apply_text_clean)
after_clean = df['text'].apply(lambda x: bool(re.search(r'\d{2,4}[-. ]\d{3,4}[-. ]\d{4}', str(x)))).sum()
print(f"\n전화번호 패턴 잔존: {before_clean}건 → {after_clean}건")

# ── 4. 40,000자 절사 ────────────────────────────────────────────────────
over_limit = (df['text'].str.len() > CHAR_LIMIT).sum()
df['text'] = df['text'].apply(lambda x: x[:CHAR_LIMIT] if len(str(x)) > CHAR_LIMIT else x)
print(f"\n{CHAR_LIMIT}자 초과 절사: {over_limit}건")

# ── 5. text_length 재계산 ──────────────────────────────────────────────
df['text_length'] = df['text'].str.len()

# ── 6. 비고 컬럼 유지 (계획서: 비고 drop 명시. 여기서는 참고용 유지) ─
# 계획서 Cell 4c: '비고 drop' — 분석에 불필요한 크롤링 메모 제거
# → 저장 시 최종 컬럼 목록에서 제외 처리 (데이터프레임에는 유지)

total = len(df)
print(f"\n전처리 후 총 레코드: {total}건")
print(f"text_length 범위: {df['text_length'].min()} ~ {df['text_length'].max()}")
print(f"\n✓ Cell 4c 완료")

page_category NaN 보완: 290건 → 0건
crawl_dual 분포:
crawl_dual
main     924
promo    234
Name: count, dtype: int64

전화번호 패턴 잔존: 47건 → 0건

40000자 초과 절사: 0건

전처리 후 총 레코드: 1158건
text_length 범위: 50 ~ 40000

✓ Cell 4c 완료


## Cell 5. 이원화 확인 (crawl_dual)

In [37]:
print("=== 이원화 분포 확인 ===")
dual_dist = df['crawl_dual'].value_counts()
print(dual_dist)

nan_check = df['crawl_dual'].isna().sum()
assert nan_check == 0, f"❌ crawl_dual NaN {nan_check}건 존재"

main_cnt = (df['crawl_dual'] == 'main').sum()
promo_cnt = (df['crawl_dual'] == 'promo').sum()
print(f"\nmain: {main_cnt}건 ({main_cnt/total*100:.1f}%)")
print(f"promo: {promo_cnt}건 ({promo_cnt/total*100:.1f}%)")
print("\n✓ Cell 5 완료 — crawl_dual 이원화 정상")

=== 이원화 분포 확인 ===
crawl_dual
main     924
promo    234
Name: count, dtype: int64

main: 924건 (79.8%)
promo: 234건 (20.2%)

✓ Cell 5 완료 — crawl_dual 이원화 정상


## Cell 6. ESG 문서 선별 (규칙 A: str.contains)

In [38]:
try:
    from tqdm import tqdm  # notebook 버전 대신 일반 버전 사용
    USE_TQDM = True
    print("tqdm 로드 완료")
except ImportError:
    def tqdm(x, **kwargs): return x
    USE_TQDM = False
    print("tqdm 없음 — 진행 표시 없이 실행")

tqdm 로드 완료


In [39]:
# ESG 키워드 str.contains 선별 + has_* 플래그 생성
hit_mask = pd.Series(False, index=df.index)

for cat, kw in tqdm(ALL_KW_FLAT, desc='ESG 선별'):
    col = f'has_{cat}'
    if col not in df.columns:
        df[col] = False
    kw_hit = df['text'].str.contains(re.escape(kw), case=False, na=False)
    df[col] = df[col] | kw_hit
    hit_mask = hit_mask | kw_hit

df['is_esg'] = hit_mask
df_esg = df[df['is_esg']].copy()

print(f"전체: {len(df)}건")
print(f"ESG 선별 (규칙A): {len(df_esg)}건")
print(f"  → 비ESG: {len(df) - len(df_esg)}건")
print(f"\nhas_* 카테고리별 문서 수:")
for cat in ESG_KEYWORDS:
    col = f'has_{cat}'
    if col in df_esg.columns:
        print(f"  {col}: {df_esg[col].sum()}건")
print("\n✓ Cell 6 완료")

ESG 선별: 100%|██████████| 257/257 [00:03<00:00, 64.27it/s]

전체: 1158건
ESG 선별 (규칙A): 1078건
  → 비ESG: 80건

has_* 카테고리별 문서 수:
  has_ESG_총괄: 430건
  has_E: 799건
  has_S: 1031건
  has_G: 269건

✓ Cell 6 완료


## Cell 7. Okt 형태소 분석 — nouns_list 생성

In [40]:
assert OKT_AVAILABLE, "❌ KoNLPy Okt 없음 — Cell 1 확인"

okt = Okt()

def extract_nouns(text):
    """Okt 명사 추출 — 1글자 순수 영문자 제거"""
    if not isinstance(text, str) or not text.strip():
        return []
    nouns = okt.nouns(text)
    return [n for n in nouns if not (len(n) == 1 and n.isalpha() and n.isascii())]

def extract_english_unigrams(text):
    """Track A raw_en용 영어 unigram 추출 — regex 기반"""
    if not isinstance(text, str) or not text.strip():
        return []
    out = []
    for m in ENGLISH_UNIGRAM_RE.finditer(text):
        tok = m.group(0).lower()
        key = normalize_alias_key(tok)
        if key in TRACKA_ENGLISH_UNIGRAM_ALIAS_KEYS:
            out.append(tok)
    return out

print("형태소 분석 시작 (시간 소요)...")
df_esg['nouns_list'] = [
    extract_nouns(t)
    for t in tqdm(df_esg['text'], desc='Okt 형태소')
]

print("영어 unigram 추출 시작 (시간 소요)...")
df_esg['english_unigrams_list'] = [
    extract_english_unigrams(t)
    for t in tqdm(df_esg['text'], desc='영어 unigram')
]

print(f"형태소 분석 완료: {len(df_esg)}건")
print(f"평균 한글 명사 수: {df_esg['nouns_list'].apply(len).mean():.1f}개")
print(f"평균 영어 unigram 수: {df_esg['english_unigrams_list'].apply(len).mean():.1f}개")
print("\n✓ Cell 7 완료")


형태소 분석 시작 (시간 소요)...


Okt 형태소: 100%|██████████| 1078/1078 [00:40<00:00, 26.69it/s]


영어 unigram 추출 시작 (시간 소요)...


영어 unigram: 100%|██████████| 1078/1078 [00:00<00:00, 3243.17it/s]

형태소 분석 완료: 1078건
평균 한글 명사 수: 371.7개
평균 영어 unigram 수: 4.7개

✓ Cell 7 완료


## Cell 8. 보정 방법 1 — main / promo 분리

In [41]:
df_esg_main = df_esg[df_esg['crawl_dual'] == 'main'].copy()
df_esg_promo = df_esg[df_esg['crawl_dual'] == 'promo'].copy()

print(f"ESG main:  {len(df_esg_main)}건")
print(f"ESG promo: {len(df_esg_promo)}건")
print(f"ESG 전체:  {len(df_esg)}건")
print("\n✓ Cell 8 완료")

ESG main:  874건
ESG promo: 204건
ESG 전체:  1078건

✓ Cell 8 완료


## Cell 9. 보정 방법 2 — ADMIN_STOPWORDS → nouns_filtered

In [42]:
def filter_nouns(noun_list):
    """행정 stopword 제거 + 1글자 제거 (ESG 키워드 포함 1글자는 보존)"""
    # ESG 키워드 중 1글자인 것 (혹시 있을 경우 보존)
    esg_1char = {kw for _, kw in ALL_KW_FLAT if len(kw) == 1}
    return [
        n for n in noun_list
        if n not in ADMIN_STOPWORDS
        and (len(n) > 1 or n in esg_1char)
    ]

def filter_english_unigrams(token_list):
    """Track A raw_en 필터 — Cell 3 영어 unigram alias 기준 유지"""
    out = []
    for tok in token_list:
        key = normalize_alias_key(tok)
        if key in TRACKA_ENGLISH_UNIGRAM_ALIAS_KEYS:
            out.append(str(tok).lower())
    return out

df_esg['nouns_filtered'] = df_esg['nouns_list'].apply(filter_nouns)
df_esg['english_unigrams_filtered'] = df_esg['english_unigrams_list'].apply(filter_english_unigrams)
df_esg['trackA_raw_total_list'] = df_esg.apply(
    lambda r: list(r['nouns_filtered']) + list(r['english_unigrams_filtered']),
    axis=1
)

before_avg = df_esg['nouns_list'].apply(len).mean()
after_avg = df_esg['nouns_filtered'].apply(len).mean()
before_en_avg = df_esg['english_unigrams_list'].apply(len).mean()
after_en_avg = df_esg['english_unigrams_filtered'].apply(len).mean()

print(f"stopword 적용 전 평균 한글 명사: {before_avg:.1f}개")
print(f"stopword 적용 후 평균 한글 명사: {after_avg:.1f}개")
print(f"한글 제거율: {(before_avg - after_avg) / before_avg * 100:.1f}%")
print(f"평균 영어 unigram (raw): {before_en_avg:.1f}개")
print(f"평균 영어 unigram (filtered): {after_en_avg:.1f}개")
print(f"평균 Track A raw_total token: {df_esg['trackA_raw_total_list'].apply(len).mean():.1f}개")
print("\n✓ Cell 9 완료")


stopword 적용 전 평균 한글 명사: 371.7개
stopword 적용 후 평균 한글 명사: 250.0개
한글 제거율: 32.8%
평균 영어 unigram (raw): 4.7개
평균 영어 unigram (filtered): 4.7개
평균 Track A raw_total token: 254.7개

✓ Cell 9 완료


## Cell 10. 보정 방법 3 — 반복 문서 제거 (대학명+계열+hash, main 우선)

In [43]:
# text hash 생성
df_esg['_text_hash'] = df_esg['text'].apply(
    lambda x: hashlib.md5(str(x).encode('utf-8')).hexdigest()
)

# main 우선 보존: main이 있는 그룹은 main을, 없으면 첫 번째 행 보존
# 정렬: crawl_dual='main' → 0, 'promo' → 1
df_esg['_dual_order'] = df_esg['crawl_dual'].map({'main': 0, 'promo': 1}).fillna(2)
df_esg_sorted = df_esg.sort_values('_dual_order')

before_dedup = len(df_esg_sorted)
df_esg_dedup = df_esg_sorted.drop_duplicates(
    subset=['대학명', '계열', '_text_hash'], keep='first'
).copy()
df_esg_dedup.drop(columns=['_dual_order'], inplace=True)

removed_count = before_dedup - len(df_esg_dedup)
df_dedup_log = df_esg_sorted[
    df_esg_sorted.duplicated(subset=['대학명', '계열', '_text_hash'], keep='first')
][['대학명', '계열', 'crawl_dual', '메뉴명', 'collected_url', 'text_length']].copy()

print(f"dedup 전: {before_dedup}건")
print(f"dedup 후: {len(df_esg_dedup)}건")
print(f"제거: {removed_count}건")
print(f"\n제거된 행 계열 분포:\n{df_dedup_log['계열'].value_counts() if len(df_dedup_log) > 0 else '없음'}")
print("\n✓ Cell 10 완료")

dedup 전: 1078건
dedup 후: 1054건
제거: 24건

제거된 행 계열 분포:
계열
식품영양    13
소비자      6
주거       3
의류       1
가정교육     1
Name: count, dtype: int64

✓ Cell 10 완료


## Cell 13. n-gram 추출 (방식 B — 문장 분리 기반)

방식 B: 원문 문장 분리 → 문장별 Okt 명사 추출 → bigram/trigram 생성  
dedup 집합 기준 실행 (df_esg_dedup)

In [44]:
assert OKT_AVAILABLE, "❌ KoNLPy Okt 없음"

def normalize_ngram_token(tok: str) -> str:
    tok = str(tok).strip()
    return tok

def keep_ngram_token(tok: str, pos: str) -> bool:
    tok = normalize_ngram_token(tok)
    if not tok:
        return False
    if tok in ADMIN_STOPWORDS:
        return False
    if tok in NGRAM_KEEP_1CHAR:
        return True
    if pos in {'Noun', 'Alpha'} and len(tok) > 1:
        return True
    return False

def extract_sent_ngrams(text, okt_inst):
    """
    문장 분리 → Okt pos 기반 토큰 추출 → bigram/trigram 리스트 반환
    - 기본은 명사/영문 토큰 유지
    - 단, Track B 복원용 1글자 형태소(NGRAM_KEEP_1CHAR)는 예외적으로 보존
    """
    if not isinstance(text, str) or not text.strip():
        return [], []

    sentences = SENT_SPLIT_RE.split(text)
    bigrams, trigrams = [], []

    for sent in sentences:
        sent = sent.strip()
        if len(sent) < 4:
            continue

        pos_seq = okt_inst.pos(sent, norm=True, stem=True)
        toks = []
        for tok, pos in pos_seq:
            tok = normalize_ngram_token(tok)
            if keep_ngram_token(tok, pos):
                toks.append(tok)

        for i in range(len(toks) - 1):
            bigrams.append(f"{toks[i]}_{toks[i+1]}")
        for i in range(len(toks) - 2):
            trigrams.append(f"{toks[i]}_{toks[i+1]}_{toks[i+2]}")

    return bigrams, trigrams

def is_ui_noise(ngram: str) -> bool:
    """동일어 반복 bigram/trigram 판단"""
    parts = [p for p in str(ngram).split('_') if p]
    if not parts:
        return True
    return len(set(parts)) == 1

def apply_ngram_replace(ngram: str) -> str:
    """형태소 분석 오류 치환"""
    return NGRAM_REPLACE.get(ngram, ngram)

print("n-gram 추출 시작 (시간 소요)...")
sent_bigrams_list = []
sent_trigrams_list = []

for text in tqdm(df_esg_dedup['text'], desc='n-gram 추출'):
    bgs, tgs = extract_sent_ngrams(text, okt)

    # 후처리 1: 형태소 치환 + UI 반복 제거 + 수동 exact stop 제거
    bgs = [
        apply_ngram_replace(b) for b in bgs
        if not is_ui_noise(b) and b not in NGRAM_STOP_EXACT
    ]
    tgs = [
        apply_ngram_replace(t) for t in tgs
        if not is_ui_noise(t)
    ]

    # 후처리 2: 영어/논문 메타 n-gram 제거
    bgs = [b for b in bgs if not is_meta_noise_ngram(b)]
    tgs = [t for t in tgs if not is_meta_noise_ngram(t)]

    sent_bigrams_list.append(bgs)
    sent_trigrams_list.append(tgs)

df_esg_dedup = df_esg_dedup.copy()
df_esg_dedup['sent_bigrams_list'] = sent_bigrams_list
df_esg_dedup['sent_trigrams_list'] = sent_trigrams_list

total_bg = sum(len(b) for b in sent_bigrams_list)
total_tg = sum(len(t) for t in sent_trigrams_list)
print(f"bigram 총 {total_bg:,}건, trigram 총 {total_tg:,}건")

sample_bg = [bg for bg_list in sent_bigrams_list[:30] for bg in bg_list[:10]]
sample_tg = [tg for tg_list in sent_trigrams_list[:30] for tg in tg_list[:10]]
print("\nbigram 샘플:", sample_bg[:10])
print("trigram 샘플:", sample_tg[:10])

print("\n✓ Cell 13 완료")

n-gram 추출 시작 (시간 소요)...


n-gram 추출: 100%|██████████| 1054/1054 [01:16<00:00, 13.81it/s]

bigram 총 260,698건, trigram 총 247,975건

bigram 샘플: ['젝트_폴리', '폴리_공간', '공간_트렌드', '트렌드_조사', '조사_한국', '한국_토지', '토지_주택', '주택_공사', '공사_고객', '고객_조사']
trigram 샘플: ['젝트_폴리_공간', '폴리_공간_트렌드', '공간_트렌드_조사', '트렌드_조사_한국', '조사_한국_토지', '한국_토지_주택', '토지_주택_공사', '주택_공사_고객', '공사_고객_조사', '고객_조사_방법론']

✓ Cell 13 완료


## Cell 13 (결과표). n-gram 요약 — summarize_ngrams

In [45]:
def summarize_ngrams(df_target, ngram_col, top_n=200):
    """
    n-gram 빈도 요약 테이블 생성.
    컬럼: ngram / raw_count / doc_freq / univ_n / series_n / is_ui_noise / keep_for_report
    """
    counter = Counter()
    doc_set = defaultdict(set)
    univ_set = defaultdict(set)
    series_set = defaultdict(set)

    for idx, (ngrams, univ, ser) in enumerate(
        zip(df_target[ngram_col], df_target['대학명'], df_target['계열'])
    ):
        for ng in ngrams:
            counter[ng] += 1
            doc_set[ng].add(idx)
            univ_set[ng].add(univ)
            series_set[ng].add(ser)

    rows = []
    for ng, cnt in counter.most_common(top_n):
        is_noise = is_ui_noise(ng) or (ng in NGRAM_STOP_EXACT)
        rows.append({
            'ngram': ng,
            'raw_count': cnt,
            'doc_freq': len(doc_set[ng]),
            'univ_n': len(univ_set[ng]),
            'series_n': len(series_set[ng]),
            'is_ui_noise': is_noise,
            'keep_for_report': (not is_noise),
        })
    return pd.DataFrame(rows)

bg_summary_all = summarize_ngrams(df_esg_dedup, 'sent_bigrams_list', top_n=200)
tg_summary_all = summarize_ngrams(df_esg_dedup, 'sent_trigrams_list', top_n=200)

df_esg_dedup_main = df_esg_dedup[df_esg_dedup['crawl_dual'] == 'main']
bg_summary_main = summarize_ngrams(df_esg_dedup_main, 'sent_bigrams_list', top_n=200)
tg_summary_main = summarize_ngrams(df_esg_dedup_main, 'sent_trigrams_list', top_n=200)

def series_top_ngrams(df_target, ngram_col, top_n_per_series=10):
    rows = []
    for ser, grp in df_target.groupby('계열'):
        ctr = Counter()
        for ngs in grp[ngram_col]:
            ctr.update(ngs)
        for ng, cnt in ctr.most_common(top_n_per_series):
            if not is_ui_noise(ng) and ng not in NGRAM_STOP_EXACT:
                rows.append({'계열': ser, 'ngram': ng, 'raw_count': cnt})
    return pd.DataFrame(rows)

bg_by_series = series_top_ngrams(df_esg_dedup, 'sent_bigrams_list', top_n_per_series=10)
tg_by_series = series_top_ngrams(df_esg_dedup, 'sent_trigrams_list', top_n_per_series=10)

print(f"bigram 요약(전체):  {len(bg_summary_all)}건")
print(f"trigram 요약(전체): {len(tg_summary_all)}건")
print(f"bigram 요약(main):  {len(bg_summary_main)}건")
print(f"trigram 요약(main): {len(tg_summary_main)}건")
print(f"계열별 bigram:      {len(bg_by_series)}건")
print(f"계열별 trigram:     {len(tg_by_series)}건")

print("\nbigram 상위 20:")
print(bg_summary_all[['ngram','raw_count','doc_freq','univ_n','is_ui_noise']].head(20).to_string())

print("\ntrigram 상위 20:")
print(tg_summary_all[['ngram','raw_count','doc_freq','univ_n','is_ui_noise']].head(20).to_string())

print("\n✓ Cell 13-결과표 완료")

bigram 요약(전체):  200건
trigram 요약(전체): 200건
bigram 요약(main):  200건
trigram 요약(main): 200건
계열별 bigram:      70건
계열별 trigram:     70건

bigram 상위 20:
         ngram  raw_count  doc_freq  univ_n  is_ui_noise
0      생활과학_대학        656       147       9        False
1       식품_영양학        621       160      14        False
2       패션_디자인        553        96      15        False
3        한국_의류        463        39      11        False
4         사회_적        372       157      16        False
5        의류_산업        342        66      12        False
6         기능_성        274       106      13        False
7   서울대학교_생활과학        250        81       6        False
8        식품_영양        248        91      13        False
9        식품_공학        244        76      12        False
10       한국_가정        241        31       6        False
11       패션_산업        239        74      12        False
12        창의_적        218       115      17        False
13       인공_지능        215        69      15        False


## Cell 14. raw ESG count + zero ESG token 검토

In [46]:
# ── 1. raw ESG count: 카테고리별 원문 키워드 등장 횟수 ─────────────────
def count_esg_raw(text, kw_list):
    """str.count 기반 카테고리 내 키워드 총 등장 횟수 + matched_kw 리스트"""
    text_lower = str(text).lower()
    total = 0
    matched = []
    for kw in kw_list:
        cnt = text_lower.count(kw.lower())
        if cnt > 0:
            total += cnt
            matched.append(kw)
    return total, matched

raw_count_rows = []
for idx, row in tqdm(df_esg_dedup.iterrows(), total=len(df_esg_dedup), desc='raw ESG count'):
    rec = {
        '대학명': row['대학명'], '계열': row['계열'],
        'crawl_dual': row['crawl_dual'], 'page_type': row['page_type'],
        'collected_url': row.get('collected_url', ''),
    }
    total_all = 0
    for cat, kws in ESG_KEYWORDS.items():
        kws_no_fp = [kw for kw in kws if kw not in FP_EXCLUDE_KW]
        cnt, matched = count_esg_raw(row['text'], kws_no_fp)
        rec[f'raw_count_{cat}'] = cnt
        rec[f'matched_kw_{cat}'] = '|'.join(matched)
        total_all += cnt
    rec['raw_count_total'] = total_all
    raw_count_rows.append(rec)

df_raw_count = pd.DataFrame(raw_count_rows)

print("raw ESG count 기술통계 (dedup 기준):")
for cat in list(ESG_KEYWORDS.keys()) + ['total']:
    col = f'raw_count_{cat}'
    if col in df_raw_count.columns:
        print(f"  {cat}: 합계={df_raw_count[col].sum()}, 평균={df_raw_count[col].mean():.2f}")

# ── 2. zero ESG token 검토 ─────────────────────────────────────────────
# nouns_filtered 기준 ESG 키워드 토큰 0개인 문서
ESG_NOUN_KW = set()
for cat, kws in ESG_KEYWORDS.items():
    for kw in kws:
        if kw not in FP_EXCLUDE_KW and len(kw) >= 2:
            ESG_NOUN_KW.add(kw)

def has_esg_noun(noun_list):
    return any(n in ESG_NOUN_KW for n in noun_list)

df_esg_dedup['has_esg_noun'] = df_esg_dedup['nouns_filtered'].apply(has_esg_noun)
df_zero = df_esg_dedup[~df_esg_dedup['has_esg_noun']].copy()

print(f"\nzero ESG token 문서: {len(df_zero)}건 / ESG 전체 {len(df_esg_dedup)}건")
if len(df_zero) > 0:
    print(f"  계열 분포:\n{df_zero['계열'].value_counts()}")
    print(f"  샘플 (상위 5건):")
    for _, r in df_zero.head(5).iterrows():
        print(f"    [{r['대학명']}/{r['계열']}] {str(r['text'])[:80]}")

print("\n✓ Cell 14 완료")

raw ESG count: 100%|██████████| 1054/1054 [00:00<00:00, 1804.05it/s]

raw ESG count 기술통계 (dedup 기준):
  ESG_총괄: 합계=7636, 평균=7.24
  E: 합계=6743, 평균=6.40
  S: 합계=28318, 평균=26.87
  G: 합계=596, 평균=0.57
  total: 합계=43293, 평균=41.07

zero ESG token 문서: 126건 / ESG 전체 1054건
  계열 분포:
계열
식품영양      60
의류        37
아동가족      13
주거        10
가정교육       4
소비자        1
생활과학대학     1
Name: count, dtype: int64
  샘플 (상위 5건):
    [이화여자대학교/소비자] 개요 학부생 대상 학회장, 부학회장, 구성 학기중 5~10회 모임 매학기 회원들의 논의를 통하여 그 학기 진행 내용과 방식을 결정함 주요 내용 
    [서울교육대학교/가정교육] 개인정보처리방침 SEOUL NATIONAL UNIVERSITY OF EDUCATION 본 웹사이트에 된 이메일 주소가 전자우편 수집 그램이나 그
    [건국대학교/주거] fnctId=profInfo,fnctNo=92 이필하 (Yi, Philha) 메일 보내기 직위 (Position) 교수 분야 (Research 
    [건국대학교/의류] fnctId=profInfo,fnctNo=590 황진숙 (Jin Sook Hwang) 메일 보내기 직위 (Position) 교수 분야 (Rese
    [건국대학교/의류] fnctId=profInfo,fnctNo=590 이명선 (Myeongseon Yi) 메일 보내기 직위 (Position) 조교수 분야 (Rese

✓ Cell 14 완료


## Cell 11. 최종 집합 확정 — 단계별 문서 수 요약

In [47]:
steps_data = [
    {'단계': '0. 원본 로드', '건수': total, '비고': 'ESG_crawled_final.jsonl'},
    {'단계': '1. page_category NaN 보완', '건수': total, '비고': f'NaN {before_nan}건 → main 보완'},
    {'단계': '2. ESG 선별 (규칙A)', '건수': len(df_esg), '비고': 'str.contains 기반'},
    {'단계': '3. dedup 제거', '건수': len(df_esg_dedup), '비고': f'{removed_count}건 제거 (대학+계열+hash, main 우선)'},
    {'단계': '4. ESG dedup main', '건수': len(df_esg_dedup_main), '비고': 'crawl_dual=main'},
    {'단계': '5. ESG dedup promo', '건수': len(df_esg_dedup[df_esg_dedup['crawl_dual']=='promo']), '비고': 'crawl_dual=promo'},
    {'단계': '6. zero ESG token', '건수': len(df_zero), '비고': 'nouns_filtered 기준'},
]
steps_df = pd.DataFrame(steps_data)
print(steps_df.to_string(index=False))
print("\n✓ Cell 11 완료")

                     단계   건수                           비고
               0. 원본 로드 1158      ESG_crawled_final.jsonl
1. page_category NaN 보완 1158           NaN 290건 → main 보완
        2. ESG 선별 (규칙A) 1078              str.contains 기반
            3. dedup 제거 1054 24건 제거 (대학+계열+hash, main 우선)
      4. ESG dedup main  866              crawl_dual=main
     5. ESG dedup promo  188             crawl_dual=promo
      6. zero ESG token  126            nouns_filtered 기준

✓ Cell 11 완료


## Cell 12. 저장 — JSONL 2개 + Excel 9시트

In [48]:
# ── 저장 컬럼 정의 (비고 제외, 분석 필요 컬럼만) ─────────────────────
SAVE_COLS = [
    '대학명', '단과대학', '계열', '메뉴명',
    'page_type', 'page_type_label', 'page_category', 'crawl_dual',
    'source_url', 'collected_url', 'domain',
    '설립구분', '성별구분',
    'text', 'text_length', 'text_len_bin',
    'is_esg', 'has_ESG_총괄', 'has_E', 'has_S', 'has_G',
    'nouns_list', 'nouns_filtered',
    'english_unigrams_list', 'english_unigrams_filtered', 'trackA_raw_total_list',
    'sent_bigrams_list', 'sent_trigrams_list',
    '_text_hash',
]
save_cols_exist = [c for c in SAVE_COLS if c in df_esg_dedup.columns]

out_all = OUTPUT_DIR / 'ESG_preprocessed_all.jsonl'
df_save_all = df_esg_dedup[save_cols_exist].copy()
for col in ['nouns_list', 'nouns_filtered', 'english_unigrams_list', 'english_unigrams_filtered',
            'trackA_raw_total_list', 'sent_bigrams_list', 'sent_trigrams_list']:
    if col in df_save_all.columns:
        df_save_all[col] = df_save_all[col].apply(json.dumps)
df_save_all.to_json(out_all, orient='records', lines=True, force_ascii=False)
print(f"저장: {out_all} ({len(df_save_all)}건)")

out_main = OUTPUT_DIR / 'ESG_preprocessed_main.jsonl'
df_save_main = df_esg_dedup_main[save_cols_exist].copy()
for col in ['nouns_list', 'nouns_filtered', 'english_unigrams_list', 'english_unigrams_filtered',
            'trackA_raw_total_list', 'sent_bigrams_list', 'sent_trigrams_list']:
    if col in df_save_main.columns:
        df_save_main[col] = df_save_main[col].apply(json.dumps)
df_save_main.to_json(out_main, orient='records', lines=True, force_ascii=False)
print(f"저장: {out_main} ({len(df_save_main)}건)")

out_xlsx = OUTPUT_DIR / 'ESG_단계I_전처리결과.xlsx'

meta_cols = [c for c in save_cols_exist
             if c not in ['nouns_list','nouns_filtered','english_unigrams_list',
                          'english_unigrams_filtered','trackA_raw_total_list',
                          'sent_bigrams_list','sent_trigrams_list','text']]

dual_summary = df_esg_dedup.groupby(['계열','crawl_dual']).size().reset_index(name='건수')

sw_df = pd.DataFrame({'stopword': sorted(ADMIN_STOPWORDS),
                      '비고': ['행정stopword'] * len(ADMIN_STOPWORDS)})
fp_df = pd.DataFrame({'stopword': sorted(FP_EXCLUDE_KW),
                      '비고': ['FP제외키워드'] * len(FP_EXCLUDE_KW)})
stopword_df = pd.concat([sw_df, fp_df], ignore_index=True)

def summarize_tracka_english_unigrams(df_target):
    counter = Counter()
    doc_set = defaultdict(set)
    univ_set = defaultdict(set)
    series_set = defaultdict(set)

    if 'english_unigrams_filtered' not in df_target.columns:
        return pd.DataFrame({'비고': ['english_unigrams_filtered 컬럼 없음']})

    for idx, (tokens, univ, ser) in enumerate(
        zip(df_target['english_unigrams_filtered'], df_target['대학명'], df_target['계열'])
    ):
        for tok in tokens:
            tok = str(tok).lower()
            counter[tok] += 1
            doc_set[tok].add(idx)
            univ_set[tok].add(univ)
            series_set[tok].add(ser)

    rows = []
    for tok, cnt in counter.most_common():
        canonical_ko = ALIAS_TO_CONCEPT.get(normalize_alias_key(tok), None)
        rows.append({
            'english_alias': tok,
            'canonical_ko': canonical_ko,
            'category': CONCEPT_TO_CATEGORY.get(canonical_ko, 'UNMAPPED') if canonical_ko else 'UNMAPPED',
            'raw_count': cnt,
            'doc_freq': len(doc_set[tok]),
            'univ_n': len(univ_set[tok]),
            'series_n': len(series_set[tok]),
        })

    if not rows:
        return pd.DataFrame({'비고': ['Track A 영어 unigram 검출 없음']})

    return pd.DataFrame(rows)

trackA_english_qa_df = summarize_tracka_english_unigrams(df_esg_dedup)

with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:
    steps_df.to_excel(writer, sheet_name='시트1_보정단계별문서수', index=False)

    if len(df_dedup_log) > 0:
        df_dedup_log.to_excel(writer, sheet_name='시트2_반복문서제거로그', index=False)
    else:
        pd.DataFrame({'비고': ['제거된 반복 문서 없음']}).to_excel(
            writer, sheet_name='시트2_반복문서제거로그', index=False)

    dual_summary.to_excel(writer, sheet_name='시트3_이원화분포', index=False)
    stopword_df.to_excel(writer, sheet_name='시트4_stopword목록', index=False)

    bg_summary_all.to_excel(writer, sheet_name='시트5_ESG_bigram_전체', index=False)
    tg_summary_all.to_excel(writer, sheet_name='시트6_ESG_trigram_전체', index=False)
    bg_summary_main.to_excel(writer, sheet_name='시트7_ESG_bigram_main', index=False)
    tg_summary_main.to_excel(writer, sheet_name='시트8_ESG_trigram_main', index=False)
    bg_by_series.to_excel(writer, sheet_name='시트9_계열별_ESG_bigram', index=False)
    tg_by_series.to_excel(writer, sheet_name='시트10_계열별_ESG_trigram', index=False)
    df_raw_count.to_excel(writer, sheet_name='시트11_raw_ESG_count', index=False)

    if len(df_zero) > 0:
        df_zero[meta_cols + ['text']].head(50).to_excel(
            writer, sheet_name='시트12_zero_ESG_token', index=False)
    else:
        pd.DataFrame({'비고': ['zero ESG token 문서 없음']}).to_excel(
            writer, sheet_name='시트12_zero_ESG_token', index=False)

    concept_map_df.to_excel(writer, sheet_name='시트13_ESG_개념매핑', index=False)
    trackA_english_qa_df.to_excel(writer, sheet_name='시트14_TrackA영문QA', index=False)

print(f"\nExcel 저장: {out_xlsx} (14시트)")
print("\n=" * 50)
print("\n✓ Cell 12 완료")


저장: C:\Users\legen\Desktop\Lab Project\ESG\v2\단계1_문서선별전처리\ESG_preprocessed_all.jsonl (1054건)
저장: C:\Users\legen\Desktop\Lab Project\ESG\v2\단계1_문서선별전처리\ESG_preprocessed_main.jsonl (866건)

Excel 저장: C:\Users\legen\Desktop\Lab Project\ESG\v2\단계1_문서선별전처리\ESG_단계I_전처리결과.xlsx (14시트)

=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=

✓ Cell 12 완료


In [49]:
df_esg_dedup.to_csv(OUTPUT_DIR / 'ESG_preprocessed_dedup.csv', index=False, encoding='utf-8-sig')
df_esg_dedup_main.to_csv(OUTPUT_DIR / 'ESG_preprocessed_main_dedup.csv', index=False, encoding='utf-8-sig')
print(f"CSV 저장: {OUTPUT_DIR / 'ESG_preprocessed_dedup.csv'}")
print(f"CSV 저장: {OUTPUT_DIR / 'ESG_preprocessed_main_dedup.csv'}")

CSV 저장: C:\Users\legen\Desktop\Lab Project\ESG\v2\단계1_문서선별전처리\ESG_preprocessed_dedup.csv
CSV 저장: C:\Users\legen\Desktop\Lab Project\ESG\v2\단계1_문서선별전처리\ESG_preprocessed_main_dedup.csv


## zero ESG tokens 저장

In [37]:
# ── zero ESG token 전체 저장 (별도 Excel) ──────────────────────────────
out_zero = OUTPUT_DIR / 'ESG_zero_token_검토.xlsx'

# 저장 컬럼 정의 (분석 검토에 유용한 컬럼 선택)
zero_save_cols = [
    '대학명', '단과대학', '계열', '메뉴명',
    'page_type', 'page_type_label', 'crawl_dual',
    '설립구분', '성별구분',
    'collected_url',
    'text_length',
    'has_E', 'has_S', 'has_G', 'has_ESG_총괄',
    'nouns_filtered',   # 어떤 명사가 추출됐는지 확인용
    'text',             # 원문 확인용
]
zero_save_cols_exist = [c for c in zero_save_cols if c in df_esg_dedup.columns]

# nouns_filtered를 가독성 있게 변환 (리스트 → 공백 구분 문자열)
df_zero_save = df_zero[zero_save_cols_exist].copy()
df_zero_save['nouns_filtered'] = df_zero_save['nouns_filtered'].apply(
    lambda x: ' '.join(parse_list(x)) if not isinstance(x, list) else ' '.join(x)
)

with pd.ExcelWriter(out_zero, engine='openpyxl') as writer:
    df_zero_save.to_excel(writer, sheet_name='zero_ESG_token_전체', index=False)

    # 계열별 요약 시트
    zero_summary = df_zero.groupby(['계열', 'page_type', 'crawl_dual']).agg(
        건수=('text_length', 'count'),
        text_length_평균=('text_length', 'mean'),
        has_E_건수=('has_E', 'sum'),
        has_S_건수=('has_S', 'sum'),
    ).reset_index()
    zero_summary['text_length_평균'] = zero_summary['text_length_평균'].round(0).astype(int)
    zero_summary.to_excel(writer, sheet_name='zero_요약', index=False)

print(f"zero ESG token 저장 완료: {out_zero}")
print(f"  전체: {len(df_zero_save)}건")
print(f"  계열별 분포:\n{df_zero['계열'].value_counts()}")

zero ESG token 저장 완료: C:\Users\legen\Desktop\Lab Project\ESG\v2\단계1_문서선별전처리\ESG_zero_token_검토.xlsx
  전체: 126건
  계열별 분포:
계열
식품영양      60
의류        37
아동가족      13
주거        10
가정교육       4
소비자        1
생활과학대학     1
Name: count, dtype: int64


In [38]:
df_zero_save.page_type.value_counts()  

page_type
ㄹ    73
ㅋ    19
ㄷ    11
ㅂ     9
ㄱ     7
ㅁ     3
ㅈ     2
ㅅ     1
ㅊ     1
Name: count, dtype: int64